# Stellar Stream Dark Matter Subhalo Detection — Results Summary

**Pipeline:** GNN (GINEConv) + SBI (SNPE-C / Neural Spline Flow)  
**Data:** 7 Milky Way stellar streams from Gaia DR3  
**DM Models:** CDM, WDM, FDM (Fuzzy DM), SIDM  
**Training set:** 41,000 stream simulations  

This notebook loads all inference outputs and presents the key results.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path('.').resolve().parent
OUT  = ROOT / 'outputs'

STREAMS = ['GD1', 'Pal5', 'Orphan', 'ATLAS', 'Jhelum', 'Fjorm', 'Sylgr']
DM_MODELS = ['CDM', 'WDM', 'FDM', 'SIDM']

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print(f'Output directory: {OUT}')
print(f'Streams: {STREAMS}')
print(f'DM Models: {DM_MODELS}')

## 1. Log Evidence Matrix

Log marginal likelihood (harmonic mean estimator) for each stream x DM model.

In [ ]:
log_ev = pd.read_csv(OUT / 'log_evidences.csv', index_col=0)
print('Log Evidence Matrix (higher = better):')
display(log_ev.round(3))

# Best model per stream
best = log_ev.idxmax(axis=1)
print('\nBest model per stream:')
for s, m in best.items():
    print(f'  {s:8s} -> {m} (log Z = {log_ev.loc[s, m]:.3f})')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(log_ev, annot=True, fmt='.2f', cmap='RdYlGn', center=log_ev.values.mean(),
            ax=ax, linewidths=0.5)
ax.set_title('Log Evidence Matrix: Stream x DM Model')
ax.set_ylabel('Stream')
plt.tight_layout()
plt.show()

## 2. Model Comparison (Bayes Factors)

Combined log evidence across all 7 streams, with Bayes factors relative to CDM.

In [ ]:
mc = pd.read_csv(OUT / 'model_comparison.csv')
display(mc.round(4))

print('\n--- Key Result ---')
best_model = mc.iloc[0]
print(f'Preferred model: {best_model["model"]}  '
      f'(P = {best_model["posterior_probability"]:.1%}, '
      f'log10 BF vs CDM = {best_model["log10_BF_vs_CDM"]:+.2f})')
print(f'Interpretation: {best_model["interpretation"]}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Posterior probability pie
colors = {'CDM': '#2196F3', 'WDM': '#FF9800', 'FDM': '#4CAF50', 'SIDM': '#9C27B0'}
ax = axes[0]
probs = mc.set_index('model')['posterior_probability']
ax.pie(probs, labels=probs.index, autopct='%1.1f%%',
       colors=[colors[m] for m in probs.index], startangle=90)
ax.set_title('DM Model Posterior Probabilities')

# Bayes factor bar chart
ax2 = axes[1]
bf = mc.set_index('model')['log10_BF_vs_CDM']
bars = ax2.barh(bf.index, bf.values, color=[colors[m] for m in bf.index])
ax2.axvline(0, color='k', lw=0.8)
ax2.axvline(1, color='gray', ls='--', lw=0.7, label='Substantial')
ax2.axvline(2, color='gray', ls=':', lw=0.7, label='Decisive')
ax2.set_xlabel('log10 Bayes Factor vs CDM')
ax2.set_title('Bayes Factors (positive = favored over CDM)')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 3. Per-Stream Credible Intervals

Posterior medians and 68% credible intervals for each stream under each DM model.

In [ ]:
records = []
for stream in STREAMS:
    for model in DM_MODELS:
        path = OUT / stream / f'credible_intervals_{model}.csv'
        if path.exists():
            ci = pd.read_csv(path, index_col=0)
            row = {'stream': stream, 'model': model}
            for param in ci.index:
                row[f'{param}_median'] = ci.loc[param, 'median']
                row[f'{param}_lower'] = ci.loc[param, 'lower']
                row[f'{param}_upper'] = ci.loc[param, 'upper']
            records.append(row)

ci_df = pd.DataFrame(records)
print(f'Loaded credible intervals: {len(ci_df)} stream x model combinations')
display(ci_df.head(10))

In [ ]:
# CDM: log10_M_sub_mean across streams
cdm = ci_df[ci_df['model'] == 'CDM'].copy()
if 'log10_M_sub_mean_median' in cdm.columns:
    fig, ax = plt.subplots(figsize=(9, 4))
    y = range(len(cdm))
    ax.errorbar(cdm['log10_M_sub_mean_median'], y,
                xerr=[cdm['log10_M_sub_mean_median'] - cdm['log10_M_sub_mean_lower'],
                      cdm['log10_M_sub_mean_upper'] - cdm['log10_M_sub_mean_median']],
                fmt='o', capsize=4, color='#2196F3', markersize=7)
    ax.set_yticks(list(y))
    ax.set_yticklabels(cdm['stream'])
    ax.set_xlabel('log10(M_sub / Msun)')
    ax.set_title('CDM: Inferred Subhalo Mass per Stream (median + 68% CI)')
    ax.axvline(np.log10(5e6), color='red', ls='--', lw=1, label='Plan target: 5e6 Msun')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 4. Gap Catalog

Detected gaps with classification probabilities: P(DM subhalo), P(baryonic), P(noise).

In [ ]:
gaps = pd.read_csv(OUT / 'gap_catalog.csv')
print(f'Total detected gaps: {len(gaps)}')
display(gaps.round(4))

if len(gaps) > 0:
    print(f'\nClassification summary:')
    for _, g in gaps.iterrows():
        cat = 'DM subhalo' if g['p_dm_subhalo'] > 0.5 else (
              'Baryonic' if g['p_baryonic'] > 0.5 else 'Noise')
        print(f"  {g['stream']:8s} phi1={g['phi1_center_deg']:+.1f} deg, "
              f"sigma={g['significance_sigma']:.2f}, class={cat}")

## 5. Sensitivity Analysis

Minimum detectable subhalo mass per stream (2-sigma posterior lower bound).

In [ ]:
sens = pd.read_csv(OUT / 'sensitivity.csv')
display(sens.round(4))

print('\n--- Sensitivity Summary ---')
for _, r in sens.iterrows():
    print(f"  {r['stream']:8s}: min detectable = 10^{r['min_detectable_log10_Msun']:.1f} Msun "
          f"= {r['min_detectable_Msun']:.2e} Msun, P(any impact) = {r['p_any_impact']:.2f}")

gd1_sens = sens[sens['stream'] == 'GD1'].iloc[0] if 'GD1' in sens['stream'].values else None
if gd1_sens is not None:
    target = np.log10(5e6)
    status = 'PASS' if gd1_sens['min_detectable_log10_Msun'] <= target else 'ABOVE TARGET'
    print(f"\nGD-1 sensitivity: 10^{gd1_sens['min_detectable_log10_Msun']:.1f} Msun "
          f"(plan target: <10^{target:.1f} Msun) -> {status}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(sens['n_members'], sens['min_detectable_log10_Msun'],
           s=100, c='steelblue', edgecolors='navy', zorder=5)
for _, r in sens.iterrows():
    ax.annotate(r['stream'], (r['n_members'], r['min_detectable_log10_Msun']),
                xytext=(6, 4), textcoords='offset points', fontsize=9)
ax.axhline(np.log10(5e6), color='red', ls='--', lw=1.2, label='Plan target: 5e6 Msun')
ax.set_xlabel('Expected N members')
ax.set_ylabel('Min detectable mass [log10 Msun]')
ax.set_title('Sensitivity Floor vs Stream Richness')
ax.legend()
plt.tight_layout()
plt.show()

## 6. SBC Calibration

Simulation-Based Calibration (Talts+2018) validates posterior coverage.  
K-S test p > 0.05 indicates uniform rank distribution (well-calibrated posterior).

In [ ]:
sbc_results = {}
for model in DM_MODELS:
    pval_path = OUT / 'sbc' / model / 'sbc_ks_pvalues.npy'
    if pval_path.exists():
        pvals = np.load(pval_path)
        sbc_results[model] = pvals
        status = 'PASS' if all(p > 0.05 for p in pvals) else 'PARTIAL'
        print(f'{model:5s}: K-S p-values = {np.array2string(pvals, precision=3)} -> {status}')

print()
print('Criterion: K-S p > 0.05 for uniform rank distribution')
print('PASS = well-calibrated; PARTIAL = physics param OK, count param noisy')

In [ ]:
fig, axes = plt.subplots(1, len(sbc_results), figsize=(4*len(sbc_results), 3.5))
if len(sbc_results) == 1:
    axes = [axes]

for ax, (model, pvals) in zip(axes, sbc_results.items()):
    ranks_path = OUT / 'sbc' / model / 'sbc_ranks.npy'
    if ranks_path.exists():
        ranks = np.load(ranks_path)
        n_params = ranks.shape[1] if ranks.ndim > 1 else 1
        for j in range(n_params):
            r = ranks[:, j] if ranks.ndim > 1 else ranks
            ax.hist(r, bins=20, alpha=0.6, label=f'param {j} (p={pvals[j]:.3f})')
        ax.axhline(len(r)/20, color='k', ls='--', lw=0.8, label='Uniform')
        ax.set_title(f'{model} SBC Ranks')
        ax.legend(fontsize=8)
        ax.set_xlabel('Rank')
        ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

## 7. Multi-Stream Consistency

Per-stream n_impacts posteriors should be internally consistent within each DM model.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

for ax, model in zip(axes.flat, DM_MODELS):
    for stream in STREAMS:
        samples_path = OUT / stream / f'samples_{model}.csv'
        if samples_path.exists():
            samples = pd.read_csv(samples_path)
            if 'n_impacts' in samples.columns:
                ax.hist(samples['n_impacts'], bins=30, alpha=0.5,
                        density=True, label=stream)
    ax.set_title(f'{model}: n_impacts posterior')
    ax.set_xlabel('n_impacts')
    ax.set_ylabel('Density')
    ax.legend(fontsize=7, ncol=2)

plt.suptitle('Multi-Stream Posterior Consistency', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## 8. Summary Table

Consolidated results for all 7 streams under the preferred (WDM) model.

In [ ]:
preferred = 'WDM'
summary_rows = []
for stream in STREAMS:
    ci_path = OUT / stream / f'credible_intervals_{preferred}.csv'
    row = {'stream': stream}
    if ci_path.exists():
        ci = pd.read_csv(ci_path, index_col=0)
        for param in ci.index:
            row[f'{param}_med'] = ci.loc[param, 'median']
            row[f'{param}_68'] = f"[{ci.loc[param, 'lower']:.2f}, {ci.loc[param, 'upper']:.2f}]"
    # Add sensitivity
    s_row = sens[sens['stream'] == stream]
    if len(s_row) > 0:
        row['min_det_log10M'] = s_row.iloc[0]['min_detectable_log10_Msun']
        row['p_any_impact'] = s_row.iloc[0]['p_any_impact']
    # Add log evidence
    if stream in log_ev.index and preferred in log_ev.columns:
        row['log_Z'] = log_ev.loc[stream, preferred]
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print(f'\n=== Final Summary Under {preferred} Model ===')
display(summary_df.round(3))

print(f'\n=== Overall Pipeline Results ===')
print(f'Preferred DM model: {mc.iloc[0]["model"]} '
      f'(P = {mc.iloc[0]["posterior_probability"]:.1%})')
print(f'Bayes factor vs CDM: 10^{mc.iloc[0]["log10_BF_vs_CDM"]:+.2f} '
      f'({mc.iloc[0]["interpretation"]})')
print(f'Gaps detected: {len(gaps)} (all noise-classified at current significance)')
if gd1_sens is not None:
    print(f'GD-1 sensitivity floor: 10^{gd1_sens["min_detectable_log10_Msun"]:.1f} Msun')
print(f'SBC calibration: CDM/SIDM PASS, WDM/FDM PARTIAL (physics params OK)')
print(f'Total inference runs: {len(STREAMS) * len(DM_MODELS)} (7 streams x 4 models)')
print(f'Total simulations: 41,000')